# 02 - S5

**Model 2 of 14 in the lineage** (see `docs/00_research_lineage.md` for the
full map). Foundation tier, same as S4D -- but MIMO instead of SISO: one
shared recurrent state that every channel writes into and reads from,
instead of `d_model` independent per-channel states.

### The recurrence

$$x_t = \bar{A} x_{t-1} + \bar{B} u_t \qquad y_t = \mathrm{Re}(C x_t) + D u_t$$

Same equation as S4D's, and that's the point -- $\bar A$ is still diagonal,
still input-independent, still discretized via zero-order hold, still
evaluated with the exact same `ssm_lab.utils.scan.parallel_scan_log` this
repo's very first model used. What changed: $x_t$ is no longer one state
per channel. It's a single $P$-dimensional ("d_state") complex vector
*shared* across all $H$ ("d_model") channels. $\bar B$ is now a real
$(P, H)$ matrix and $\bar C$ a real $(H, P)$ matrix -- both genuine
matrix-vector products, not S4D's elementwise per-channel scaling. Every
channel's input gets mixed into the shared state; every channel's output
reads a mix of the whole state back out. Cross-channel mixing happens
*inside* the recurrence now, not only in the FFN around it.

**References:** Smith, Warrington & Linderman, *Simplified State Space
Layers for Sequence Modeling* (S5, ICLR 2023); Gu et al., *HiPPO: Recurrent
Memory with Optimal Polynomial Projections* (NeurIPS 2020, same foundation
S4D also builds on).

**Honest scope note:** this repo's S5 reuses S4D's diagonal HiPPO-LegS
approximation (`ssm_lab.utils.hippo.hippo_diag_spectrum`) for $A$'s init,
rather than rederiving S5's own specific eigendecomposition-of-a-normal-
approximation -- both are diagonal approximations to the same underlying
spectrum; see the docstring in `src/ssm_lab/layers/s5.py` and
`docs/02_s5.md` for the full reasoning. The property that matters most for
everything below: $\bar A$ is *still* a learned constant, same as S4D --
MIMO mixing is a different axis entirely from selectivity, and this
notebook's predictions (bottom) are built on keeping those two axes
straight.

## Setup

Run this cell first, every time. It detects Colab vs. Kaggle vs. local,
gets `ssm_lab` importable either way, and installs dependencies if needed.
Fill in `REPO_URL` below once you've pushed this repo to your own GitHub.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/YOUR_USERNAME/state-space-models.git"  # <- set this after pushing


def _in_colab() -> bool:
    return "google.colab" in sys.modules


def _in_kaggle() -> bool:
    return "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle")


IN_COLAB, IN_KAGGLE = _in_colab(), _in_kaggle()
IN_HOSTED_NOTEBOOK = IN_COLAB or IN_KAGGLE
print(f"Environment -> Colab: {IN_COLAB} | Kaggle: {IN_KAGGLE} | Local: {not IN_HOSTED_NOTEBOOK}")


def ensure_repo_on_path() -> None:
    try:
        import ssm_lab  # noqa: F401

        print("ssm_lab already importable -- nothing to do.")
        return
    except ImportError:
        pass

    for candidate in ["state-space-models", "/kaggle/working/state-space-models", "."]:
        if os.path.isdir(os.path.join(candidate, "src", "ssm_lab")):
            sys.path.insert(0, os.path.join(candidate, "src"))
            print(f"Found repo at '{candidate}', added src/ to sys.path.")
            return

    if IN_HOSTED_NOTEBOOK:
        print("Repo not found locally -- cloning...")
        subprocess.run(["git", "clone", "--quiet", REPO_URL], check=True)
        sys.path.insert(0, "state-space-models/src")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", "state-space-models/requirements.txt"],
            check=True,
        )
    else:
        raise RuntimeError(
            "Could not find ssm_lab. Run this notebook from inside the repo "
            "(e.g. `jupyter notebook` from the repo root), or set REPO_URL "
            "above and re-run."
        )


ensure_repo_on_path()
import ssm_lab  # noqa: E402

print("ssm_lab imported OK from:", ssm_lab.__file__)


In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))
else:
    print(
        "No GPU detected. On Kaggle: Settings -> Accelerator -> GPU T4 x2 or P100. "
        "On Colab: Runtime -> Change runtime type -> T4 GPU. "
        "CPU works fine for FAST_DEV_RUN=True."
    )


## Fast dev run

`FAST_DEV_RUN = True` shrinks every config below to a ~10-30 second CPU-friendly
sanity check (tiny model, tiny sequences, ~100 steps) -- use it to confirm the
whole notebook runs cleanly before committing GPU time to it. Set it to
`False` for the real run this notebook is meant to produce.

In [ ]:
FAST_DEV_RUN = True  # set False for the real run


## Shared run helper

Identical to `01_s4.ipynb`'s, with `S4DLayer` swapped for `S5Layer` -- the
entire point of the shared `SequenceMixer` interface is that nothing else
about this function needs to change.

In [ ]:
import json
import time

import matplotlib.pyplot as plt

from ssm_lab.layers.s5 import S5Layer
from ssm_lab.models.backbone import BackboneConfig, SequenceBackbone
from ssm_lab.training.trainer import Trainer, TrainConfig

MODEL_NAME = "s5"
ALL_RESULTS = {}  # populated by every section below, saved to results/ at the end


def run_benchmark(
    tag, vocab_size, train_fn, val_fn,
    d_model, d_state, n_layers, max_steps, lr=3e-3, eval_iters=10, plot=True, seed=0,
):
    torch.manual_seed(seed)  # reseed here so results don't depend on how much randomness earlier cells consumed
    cfg = BackboneConfig(vocab_size=vocab_size, d_model=d_model, n_layers=n_layers, mixer_kwargs=dict(d_state=d_state))
    model = SequenceBackbone(S5Layer, cfg)
    tcfg = TrainConfig(
        lr=lr, warmup_steps=max(10, max_steps // 15), max_steps=max_steps,
        eval_every=max(10, max_steps // 10), eval_iters=eval_iters,
        log_every=max(10, max_steps // 5), device=DEVICE,
    )
    trainer = Trainer(model, tcfg, run_name=f"{MODEL_NAME}_{tag}")
    t0 = time.time()
    history = trainer.fit(train_fn, val_fn)
    elapsed = time.time() - t0

    result = {
        "params": model.num_params(),
        "final_val_loss": history["val_loss"][-1],
        "final_val_acc": history["val_acc"][-1],
        "train_seconds": elapsed,
    }
    ALL_RESULTS[tag] = result
    print(f"[{tag}] params={result['params']:,}  val_acc={result['final_val_acc']:.3f}  "
          f"val_loss={result['final_val_loss']:.3f}  ({elapsed:.1f}s)")

    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(9, 3))
        axes[0].plot(history["step"], history["val_loss"], marker="o")
        axes[0].set(title=f"{tag}: val loss", xlabel="step")
        axes[1].plot(history["step"], history["val_acc"], marker="o", color="darkorange")
        axes[1].set(title=f"{tag}: val accuracy", xlabel="step", ylim=(0, 1.02))
        fig.tight_layout()
        plt.show()
    return history


## A note on `d_state` before running anything

S5's `d_state` is a **shared** budget across every channel -- S4D's
`d_state=64` at `d_model=128` means 128 x 64 = 8,192 total complex numbers
of state; S5's `d_state=64` at the same `d_model` means just 64. This
notebook does **not** attempt to capacity-match the two architectures
exactly (that's a question for the eventual cross-model comparison, once
more models exist) -- it uses generously-sized, reasonable `d_state`
values for S5 (larger than S4D's, since S5's scan cost doesn't scale with
`d_model` the way S4D's does -- see §6) and focuses on whether the *same
qualitative failure modes* show up, not on precisely matched capacity.

## 1. MQAR (associative recall)

S5's transition is **just as non-selective as S4D's** -- $\bar A$ is a
learned constant either way. MIMO mixing changes *how much of the state a
given piece of information can spread across*, not *whether the model can
decide what's worth remembering in the first place*. Prediction: S5 should
be weak here too, for the same structural reason as S4D, not meaningfully
better -- if it is meaningfully better, that's a genuinely interesting
result worth flagging rather than expecting, since it would suggest
cross-channel capacity sharing matters more for recall than this repo's
working model of "selectivity is the whole story" assumes.

In [ ]:
from ssm_lab.benchmarks.mqar import make_mqar_loader

if FAST_DEV_RUN:
    mqar_cfg = dict(batch_size=32, seq_len=32, vocab_size=32, num_kv_pairs=4)
    mqar_train_steps, mqar_d_model, mqar_d_state = 200, 48, 96
else:
    mqar_cfg = dict(batch_size=64, seq_len=256, vocab_size=8192, num_kv_pairs=32)
    mqar_train_steps, mqar_d_model, mqar_d_state = 2000, 128, 256

mqar_train = make_mqar_loader(**mqar_cfg, device=DEVICE, seed=1)
mqar_val = make_mqar_loader(**mqar_cfg, device=DEVICE, seed=2)
_ = run_benchmark("mqar", mqar_cfg["vocab_size"], mqar_train, mqar_val,
                   d_model=mqar_d_model, d_state=mqar_d_state, n_layers=2,
                   max_steps=mqar_train_steps)


## 2. Copying

Same Jelassi et al. (2024) capacity-cliff logic as S4D's notebook, with one
twist worth watching for: because S5's whole `d_state` is shared across
channels rather than multiplied by `d_model`, and because copied content
has to route through the *same* shared state every other channel is also
using, the cliff could plausibly land at a different relative `copy_len`
than S4D's did at a nominally "equivalent" `d_state` -- another point this
notebook surfaces rather than assumes.

Same `FAST_DEV_RUN` noise caveat as `01_s4.ipynb`: a single-seed, truncated
sweep can dent monotonicity from optimization variance alone -- treat the
dev-run version as a sanity check, not the result to report.

In [ ]:
from ssm_lab.benchmarks.copying import make_copy_loader

copy_lengths = [2, 4, 8] if FAST_DEV_RUN else [4, 8, 16, 32, 64]
copy_steps = 350 if FAST_DEV_RUN else 1200
copy_d_model, copy_d_state = (32, 64) if FAST_DEV_RUN else (128, 128)  # d_state fixed across the sweep on purpose
copy_vocab = 16 if FAST_DEV_RUN else 64

copy_cliff = []
for copy_len in copy_lengths:
    seq_len = 4 * copy_len  # generous headroom + a little filler after the second copy
    train_fn = make_copy_loader(32, seq_len, copy_vocab, copy_len, device=DEVICE, seed=1)
    val_fn = make_copy_loader(32, seq_len, copy_vocab, copy_len, device=DEVICE, seed=2)
    hist = run_benchmark(f"copy_len{copy_len}", copy_vocab, train_fn, val_fn,
                          d_model=copy_d_model, d_state=copy_d_state, n_layers=2,
                          max_steps=copy_steps, plot=False)
    copy_cliff.append((copy_len, hist["val_acc"][-1]))

fig, ax = plt.subplots(figsize=(6, 3.5))
xs, ys = zip(*copy_cliff)
ax.plot(xs, ys, marker="o")
ax.set(title=f"S5 copying capacity cliff (d_state={copy_d_state} fixed, shared not per-channel)",
       xlabel="copy_len (tokens to copy)", ylabel="val accuracy", ylim=(0, 1.02))
ax.axvline(copy_d_state, linestyle="--", color="gray", alpha=0.6, label="d_state")
ax.legend()
plt.show()


## 3. Induction heads

Same prediction as MQAR, same reasoning: induction needs "notice this
specific token, remember it, retrieve it on cue," which is a selectivity
problem, not a mixing problem. S5's MIMO state doesn't add a mechanism for
deciding *what's worth writing down* -- it only changes how what gets
written down is shared across channels.

In [ ]:
from ssm_lab.benchmarks.induction_heads import make_induction_loader

if FAST_DEV_RUN:
    ind_cfg = dict(batch_size=32, seq_len=24, vocab_size=16, num_triggers=1)
    ind_steps, ind_d_model, ind_d_state = 200, 32, 64
else:
    ind_cfg = dict(batch_size=64, seq_len=256, vocab_size=64, num_triggers=3)
    ind_steps, ind_d_model, ind_d_state = 1500, 128, 256

ind_train = make_induction_loader(**ind_cfg, device=DEVICE, seed=1)
ind_val = make_induction_loader(**ind_cfg, device=DEVICE, seed=2)
_ = run_benchmark("induction", ind_cfg["vocab_size"], ind_train, ind_val,
                   d_model=ind_d_model, d_state=ind_d_state, n_layers=2, max_steps=ind_steps)


## 4. State tracking: parity and S5 (group) composition

Naming collision worth flagging explicitly: "S5" the architecture (this
notebook) and "S5" the permutation group used in the state-tracking
benchmark (`ssm_lab.benchmarks.state_tracking`, unrelated -- it's the
symmetric group on 5 elements) are two different things that happen to
share a name.

Merrill, Petty & Sabharwal (2024): a **diagonal, decay-only** linear
recurrence is stuck in TC^0 regardless of how much data or training it
gets. S5-the-architecture's transition is still exactly that shape -- MIMO
mixing doesn't add nonlinearity to the *recurrence itself* (the FFN around
it is nonlinear, but the state update `x_t = Ā x_{t-1} + B̄ u_t` is purely
linear, same as S4D's). Prediction: S5 should fail S5-tracking for the
same structural reason S4D does, not resolve it.

In [ ]:
from ssm_lab.benchmarks.state_tracking import (
    S5_VOCAB_SIZE,
    make_parity_loader,
    make_s5_tracking_loader,
)

parity_steps = 200 if FAST_DEV_RUN else 1000
seq_len_parity = 16 if FAST_DEV_RUN else 64
_ = run_benchmark("parity", 2,
                   make_parity_loader(32, seq_len_parity, device=DEVICE, seed=1),
                   make_parity_loader(32, seq_len_parity, device=DEVICE, seed=2),
                   d_model=32 if FAST_DEV_RUN else 96, d_state=64 if FAST_DEV_RUN else 192,
                   n_layers=2, max_steps=parity_steps)

s5group_steps = 200 if FAST_DEV_RUN else 1200
seq_len_s5group = 10 if FAST_DEV_RUN else 32
_ = run_benchmark("s5_tracking", S5_VOCAB_SIZE,
                   make_s5_tracking_loader(32, seq_len_s5group, device=DEVICE, seed=1),
                   make_s5_tracking_loader(32, seq_len_s5group, device=DEVICE, seed=2),
                   d_model=48 if FAST_DEV_RUN else 128, d_state=96 if FAST_DEV_RUN else 256,
                   n_layers=2, max_steps=s5group_steps)


## 5. Real language modeling

Same grounding purpose as `01_s4.ipynb` section 5 -- synthetic benchmarks
isolate *why*, this section checks *whether it works as language at all*,
on the exact same `ssm_lab.data` pipeline (tokenize once, memmap-backed
loading, RAM usage independent of corpus size).

In [ ]:
import math

from ssm_lab.data.tokenizer import ByteTokenizer
from ssm_lab.data.text_lm import download_tiny_shakespeare, prepare_corpus, TextLMData

corpus_path = download_tiny_shakespeare("data/tinyshakespeare.txt")
tokenizer = ByteTokenizer()
with open(corpus_path) as f:
    corpus_text = f.read()

corpus_meta = prepare_corpus(corpus_text, tokenizer, out_dir="data/tinyshakespeare_bin")
print(corpus_meta)
lm_data = TextLMData.from_bin_dir("data/tinyshakespeare_bin")

if FAST_DEV_RUN:
    lm_steps, lm_d_model, lm_d_state, lm_batch, lm_seq_len = 200, 48, 96, 16, 48
else:
    lm_steps, lm_d_model, lm_d_state, lm_batch, lm_seq_len = 3000, 256, 256, 64, 256

lm_train = lm_data.get_batch_fn(lm_batch, lm_seq_len, split="train", device=DEVICE, seed=1)
lm_val = lm_data.get_batch_fn(lm_batch, lm_seq_len, split="val", device=DEVICE, seed=2)
lm_hist = run_benchmark("real_lm_tinyshakespeare", lm_data.vocab_size, lm_train, lm_val,
                         d_model=lm_d_model, d_state=lm_d_state, n_layers=3, max_steps=lm_steps, plot=False)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot(lm_hist["step"], lm_hist["val_loss"], marker="o")
axes[0].set(title="real LM: val loss (nats/byte)", xlabel="step")
val_ppl = [math.exp(l) for l in lm_hist["val_loss"]]
axes[1].plot(lm_hist["step"], val_ppl, marker="o", color="darkorange")
axes[1].set(title="real LM: val perplexity", xlabel="step")
axes[1].axhline(lm_data.vocab_size, linestyle="--", color="gray", alpha=0.6, label="chance (uniform)")
axes[1].legend()
fig.tight_layout()
plt.show()

print(f"chance perplexity (uniform over {lm_data.vocab_size} bytes): {lm_data.vocab_size}")
print(f"final val perplexity: {val_ppl[-1]:.2f}")


## 6. Throughput vs. sequence length

The other prediction worth stating before running this: S5's scan cost is
$O(L \log L \cdot P)$ -- independent of $d_{model}$ -- while S4D's is
$O(L \log L \cdot d_{model} \cdot d_{state})$. At a `d_state` chosen to be
smaller than `d_model x S4D's d_state`, S5's scan itself should be cheaper
per layer; the `B@u_t` / `C@h_t` matrix-vector products add a comparable
cost to a linear layer on top of that, which S4D doesn't need since its
mixing is elementwise. Whether that nets out faster or slower than S4D
at any particular config is an empirical question this cell actually
answers, not a foregone conclusion.

In [ ]:
seq_lengths = [64, 128, 256] if FAST_DEV_RUN else [128, 256, 512, 1024, 2048, 4096]
bench_d_model, bench_n_layers, bench_d_state = (32, 2, 64) if FAST_DEV_RUN else (128, 4, 256)
n_repeats = 3

bench_model = SequenceBackbone(
    S5Layer,
    BackboneConfig(vocab_size=256, d_model=bench_d_model, n_layers=bench_n_layers, mixer_kwargs=dict(d_state=bench_d_state)),
).to(DEVICE).eval()

timings = []
with torch.no_grad():
    for L in seq_lengths:
        x = torch.randint(0, 256, (8, L), device=DEVICE)
        for _ in range(2):  # warmup
            bench_model(x)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_repeats):
            bench_model(x)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        dt = (time.time() - t0) / n_repeats
        timings.append(dt)
        print(f"seq_len={L:5d}  forward time={dt*1000:8.1f} ms")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(seq_lengths, timings, marker="o")
ax.set(title=f"S5 forward-pass time vs sequence length ({DEVICE})",
       xlabel="sequence length", ylabel="seconds/forward pass")
plt.show()

ALL_RESULTS["throughput"] = {"seq_lengths": seq_lengths, "forward_seconds": timings, "device": DEVICE}


## 7. Scaling infrastructure: checkpointing, accumulation, resume

Same shared infrastructure as `01_s4.ipynb` section 7 -- gradient
checkpointing, gradient accumulation, and exact checkpoint/resume are all
properties of `SequenceBackbone`/`Trainer`, not of any one mixer, so they
apply to S5 completely unchanged. Demonstrated again here (not just
described) for the same reason as before: a fresh, differently-initialized
model reloaded from a checkpoint should reproduce bit-identical eval loss
on a fixed batch, then keep training from where it left off.

In [ ]:
from ssm_lab.models.presets import get_preset

scale_steps = 100 if FAST_DEV_RUN else 2000
preset_name = "tiny" if FAST_DEV_RUN else "small"
scale_cfg, scale_train_overrides = get_preset(preset_name, vocab_size=lm_data.vocab_size)
print(f"preset '{preset_name}': {scale_cfg}, train_overrides={scale_train_overrides}")

scale_train = lm_data.get_batch_fn(16, 48, split="train", device=DEVICE, seed=3)
scale_val = lm_data.get_batch_fn(16, 48, split="val", device=DEVICE, seed=4)

torch.manual_seed(0)
model_session1 = SequenceBackbone(S5Layer, scale_cfg)
tcfg_session1 = TrainConfig(max_steps=scale_steps, eval_every=scale_steps, eval_iters=5,
                             log_every=scale_steps, device=DEVICE, **scale_train_overrides)
trainer_session1 = Trainer(model_session1, tcfg_session1, run_name="scaling_demo_s5")
trainer_session1.fit(scale_train, scale_val)

trainer_session1.save_checkpoint("checkpoints/scaling_demo_s5.pt")
print(f"session 1 done at step {trainer_session1.global_step}, checkpoint saved.")

fixed_eval_batch = scale_val()
fixed_eval_fn = lambda: fixed_eval_batch  # noqa: E731
loss_before_reload, _ = trainer_session1.evaluate(fixed_eval_fn, n_iters=1)

# --- simulate a fresh session: brand-new model, differently initialized ---
torch.manual_seed(999)
model_session2 = SequenceBackbone(S5Layer, scale_cfg)
trainer_session2 = Trainer(model_session2, tcfg_session1, run_name="scaling_demo_s5")
trainer_session2.load_checkpoint("checkpoints/scaling_demo_s5.pt")
loss_after_reload, _ = trainer_session2.evaluate(fixed_eval_fn, n_iters=1)

print(f"global_step restored: {trainer_session2.global_step} (expected {trainer_session1.global_step})")
print(f"eval loss on a fixed batch, before save:  {loss_before_reload:.6f}")
print(f"eval loss on that SAME fixed batch, after reload into a differently-initialized fresh model: {loss_after_reload:.6f}")
assert abs(loss_before_reload - loss_after_reload) < 1e-5, "checkpoint restoration should be exact"
print("exact match confirmed.")

trainer_session2.fit(scale_train, scale_val, max_steps=trainer_session2.global_step + max(20, scale_steps // 5))
print(f"resumed training to step {trainer_session2.global_step} without restarting from 0.")


## Summary

Saved to `results/s5_results.json` -- same key names as
`results/s4d_results.json`, so `20_benchmark_suite.ipynb` can load both
generically once it exists.

In [ ]:
import os

print(f"{'benchmark':<16}{'val_acc':>10}{'val_loss':>12}{'params':>12}")
for tag, r in ALL_RESULTS.items():
    if tag == "throughput":
        continue
    print(f"{tag:<16}{r['final_val_acc']:>10.3f}{r['final_val_loss']:>12.3f}{r['params']:>12,}")

os.makedirs("results", exist_ok=True)
out_path = "results/s5_results.json" if os.path.isdir("results") else "../results/s5_results.json"
with open(out_path, "w") as f:
    json.dump(ALL_RESULTS, f, indent=2)
print(f"\nsaved -> {out_path}")


## What to expect (and what it means if you see it)

S5 shares S4D's core limitation -- $\bar A$ is still a learned constant,
not a function of the input -- so most predictions here mirror S4D's, on
purpose. MIMO mixing is a different axis from selectivity, not a fix for
it:

- **Copying**: expect a capacity cliff again, same as S4D, possibly at a
  different relative `copy_len` given `d_state` means something different
  here (shared, not per-channel) -- see the note before section 2.
- **MQAR / induction heads**: expected to be weak, for the same structural
  reason as S4D -- no content-based routing either way. If S5 is
  *meaningfully* better here, that's a genuinely interesting result worth
  flagging, not the expected outcome.
- **Parity / S5 (group) tracking**: expected to be weak-to-failing, same
  Merrill et al. TC^0 ceiling as S4D -- the recurrence itself is still
  linear and diagonal, regardless of MIMO mixing.
- **Real language modeling**: should work, likely comparably to S4D at
  similar parameter budgets, possibly more parameter-efficient given
  cross-channel state sharing -- an open question this section's numbers
  actually answer.
- **Throughput**: the one axis MIMO plausibly *does* move directly --
  S5's scan cost doesn't scale with `d_model`, S4D's does. Whether that
  wins out over the added cost of `B@u_t`/`C@h_t` matrix-vector products
  is what section 6 measures, not assumes.

If S5 looks like "S4D with different numbers" on the first four benchmarks
and something genuinely different on the fifth, that's the model working
as expected -- MIMO vs. SISO and selective vs. non-selective are
orthogonal design axes, and this notebook is what makes that concrete
instead of asserted. Model 3 (Mamba) is where the selectivity axis finally
moves; full reasoning in `docs/02_s5.md`.